# Algoritmo Genético — exemplo com várias classes (ordinal)

Este notebook usa o `dados_multiclasse.xlsx`, que tem **5 classes**: D, G, M, P e S.

Com várias classes o programa usa o `fitness_ordinal`. Ele precisa que as classes tenham uma **ordem** (da menor para a maior), porque o modelo dá uma nota para cada amostra e divide essa nota em faixas, uma faixa por classe.

Como a ordem real das classes ainda não é conhecida, o notebook **testa todas as ordens possíveis** e usa a que deu o melhor fitness. Quando souber a ordem certa, é só colocá-la na variável `ORDEM_CLASSES` (seção 3).

## 0. Imports

In [1]:
import sys
sys.path.insert(0, "../../src")  # pasta onde fica o pacote algoritmo_genetico

import itertools
import time

import numpy as np

from algoritmo_genetico.leitura import ler_excel
from algoritmo_genetico.criacao import criar_cromossomos
from algoritmo_genetico.fitness import fitness_ordinal
from algoritmo_genetico import rodar, prever

np.random.seed(0)  # semente: rodando de novo, os sorteios saem iguais

## 1. Lendo os dados

In [2]:
X, y_letras = ler_excel("dados_multiclasse.xlsx")
X = X.astype(float)

classes = sorted(set(y_letras))

print("X (amostras, features):", X.shape)
print("classes:", classes)
for c in classes:
    print(f"  {c}: {sum(y_letras == c)} amostras")

X (amostras, features): (261, 30)
classes: ['D', 'G', 'M', 'P', 'S']
  D: 37 amostras
  G: 61 amostras
  M: 39 amostras
  P: 64 amostras
  S: 60 amostras


## 2. Separando treino e teste

80% para treinar e 20% para testar. A escolha da ordem das classes (seção 3) usa **só o treino**, para o teste continuar sendo um dado que o modelo nunca viu.

In [3]:
ordem_amostras = np.random.permutation(len(y_letras))  # numeros de 0 a n-1 embaralhados
corte = int(0.8 * len(y_letras))

X_treino, y_treino_letras = X[ordem_amostras[:corte]], y_letras[ordem_amostras[:corte]]
X_teste, y_teste_letras = X[ordem_amostras[corte:]], y_letras[ordem_amostras[corte:]]

print("treino:", X_treino.shape, "| teste:", X_teste.shape)

treino: (208, 30) | teste: (53, 30)


## 3. Como funciona o modelo ordinal

Cada cromossomo calcula uma nota para a amostra: `nota = bias + soma(peso × feature)`.

Os **thresholds** (pontos de corte) dividem a nota em faixas. Com 5 classes precisamos de 4 cortes:

```
        classe 0  |  classe 1  |  classe 2  |  classe 3  |  classe 4
  ... ----------- -1.5 ------- -0.5 ------- 0.5 ------- 1.5 ----------- ...
```

Por dentro, o programa usa `np.digitize(nota, thresholds)`, que só diz em qual faixa a nota caiu (0, 1, 2, ...). Exemplo:

In [4]:
THRESHOLDS = np.array([-1.5, -0.5, 0.5, 1.5])

notas = np.array([-3.0, -1.0, 0.0, 1.0, 2.7])
print("notas :", notas)
print("faixas:", np.digitize(notas, THRESHOLDS))

notas : [-3.  -1.   0.   1.   2.7]
faixas: [0 1 2 3 4]


O fitness é o **produto do recall de cada classe**: se o modelo erra todas as amostras de uma classe, o fitness vira 0. Isso força o modelo a acertar um pouco de todas as classes.

### Escolhendo a ordem das classes

Se souber a ordem real, coloque aqui, da menor para a maior, por exemplo `ORDEM_CLASSES = ["D", "M", "S", "G", "P"]`.
Deixando `None`, a próxima célula testa todas as ordens.

In [5]:
ORDEM_CLASSES = None

### Busca da melhor ordem

`itertools.permutations` (do próprio Python) gera todas as ordens possíveis das 5 classes (5! = 120).
Uma ordem e a mesma ordem invertida dão o mesmo resultado (só inverte o sinal dos pesos), então testamos só 60.

Para cada ordem, roda o algoritmo com 2 sementes diferentes e tira a média do fitness no **treino**. Demora uns segundos.

In [6]:
def converter(y_letras, ordem):
    """Troca cada letra pela posicao dela na ordem (ex: ordem D,M,S -> D=0, M=1, S=2)."""
    return np.array([ordem.index(letra) for letra in y_letras])


if ORDEM_CLASSES is None:
    inicio = time.time()
    resultados = []

    for ordem in itertools.permutations(classes):
        if ordem[0] > ordem[-1]:  # pula a ordem invertida
            continue

        y_treino = converter(y_treino_letras, list(ordem))
        fitnesses = []
        for semente in [0, 1]:
            np.random.seed(semente)
            m = rodar(X_treino, y_treino, thresholds=THRESHOLDS, geracoes=500, fitness_alvo=1.0, verbose=False)
            fitnesses.append(m["fitness"])

        resultados.append((sum(fitnesses) / len(fitnesses), list(ordem)))

    resultados.sort(reverse=True)  # maior fitness primeiro
    print(f"testou {len(resultados)} ordens em {time.time() - inicio:.1f}s\n")
    print("5 melhores ordens:")
    for fitness_medio, ordem in resultados[:5]:
        print(f"  {' < '.join(ordem)}   fitness medio = {fitness_medio:.4f}")
    print("\npior ordem:")
    print(f"  {' < '.join(resultados[-1][1])}   fitness medio = {resultados[-1][0]:.4f}")

    ORDEM_CLASSES = resultados[0][1]

print("\nordem usada:", " < ".join(ORDEM_CLASSES))

testou 60 ordens em 12.8s

5 melhores ordens:
  P < D < G < M < S   fitness medio = 0.3737
  D < P < G < M < S   fitness medio = 0.3678
  D < G < P < M < S   fitness medio = 0.3532
  D < M < S < G < P   fitness medio = 0.3463
  G < P < M < D < S   fitness medio = 0.3409

pior ordem:
  G < P < S < D < M   fitness medio = 0.0518

ordem usada: P < D < G < M < S


> As melhores ordens costumam ficar com fitness parecido entre si, então a ganhadora pode mudar com outras sementes. A diferença entre as melhores e as piores é bem maior, então a busca já elimina as ordens ruins.

## 4. Convertendo as classes para números

In [7]:
y_treino = converter(y_treino_letras, ORDEM_CLASSES)
y_teste = converter(y_teste_letras, ORDEM_CLASSES)

for numero, letra in enumerate(ORDEM_CLASSES):
    print(f"{letra} -> {numero}")

P -> 0
D -> 1
G -> 2
M -> 3
S -> 4


## 5. Olhando o fitness de uma população aleatória

Antes de treinar: cromossomos aleatórios quase sempre têm fitness 0 ou muito baixo, porque é difícil acertar um pouco de **todas** as 5 classes por sorte.

In [8]:
np.random.seed(0)
populacao = criar_cromossomos(qtd_cromossomos=6, qtd_genes=X_treino.shape[1] + 1)
fitnesses = fitness_ordinal(populacao, X_treino, y_treino, THRESHOLDS)

for i, f in enumerate(fitnesses):
    print(f"cromossomo {i}: fitness = {f:.6f}")

cromossomo 0: fitness = 0.000000
cromossomo 1: fitness = 0.000000
cromossomo 2: fitness = 0.000000
cromossomo 3: fitness = 0.000000
cromossomo 4: fitness = 0.001290
cromossomo 5: fitness = 0.000226


## 6. Treinando com `rodar`

`fitness_alvo=1.0` faz rodar todas as gerações (com 5 classes, 1.0 seria acertar tudo).
Use `verbose=True` para ver o melhor fitness de cada geração.

In [9]:
np.random.seed(0)
inicio = time.time()

modelo = rodar(
    X_treino, y_treino,
    qtd_cromossomos=6,
    qtd_genes_mutados=1,
    qtd_filhos=3,
    geracoes=2000,
    fitness_alvo=1.0,
    thresholds=THRESHOLDS,
    verbose=False,
)

print(f"tempo: {time.time() - inicio:.1f}s")
print("melhor fitness:", round(modelo["fitness"], 4))
print("geracoes:", modelo["geracao"])

tempo: 0.4s
melhor fitness: 0.295
geracoes: 2000


## 7. Testando com `prever`

In [10]:
previsto = prever(modelo, X_teste)

acertos = sum(previsto == y_teste)
print(f"acuracia no teste: {acertos}/{len(y_teste)} = {acertos / len(y_teste):.2%}")

# para comparar: chutar sempre a classe mais comum do treino
mais_comum = max(range(len(ORDEM_CLASSES)), key=lambda k: sum(y_treino == k))
chute = sum(y_teste == mais_comum) / len(y_teste)
print(f"chutando sempre '{ORDEM_CLASSES[mais_comum]}': {chute:.2%}")

print("\nrecall por classe (quanto de cada classe ele acertou):")
for k, letra in enumerate(ORDEM_CLASSES):
    total = sum(y_teste == k)
    certos = sum((y_teste == k) & (previsto == k))
    print(f"  {letra}: {certos}/{total} = {certos / total:.0%}")

acuracia no teste: 42/53 = 79.25%
chutando sempre 'P': 22.64%

recall por classe (quanto de cada classe ele acertou):
  P: 9/12 = 75%
  D: 11/14 = 79%
  G: 10/11 = 91%
  M: 2/4 = 50%
  S: 10/12 = 83%


### Matriz de confusão

Cada linha é a classe real, cada coluna é a classe que o modelo previu. A diagonal são os acertos.
Como as classes são ordenadas, erros perto da diagonal (classe vizinha) são "menos graves" que erros longe dela.

In [11]:
print("real \\ previu " + "".join(f"{letra:>5}" for letra in ORDEM_CLASSES))
for real, letra_real in enumerate(ORDEM_CLASSES):
    linha = [sum((y_teste == real) & (previsto == p)) for p in range(len(ORDEM_CLASSES))]
    print(f"{letra_real:>13} " + "".join(f"{n:>5}" for n in linha))

real \ previu     P    D    G    M    S
            P     9    0    3    0    0
            D     3   11    0    0    0
            G     0    0   10    0    1
            M     0    2    0    2    0
            S     0    1    1    0   10


### Classificando uma amostra só

In [12]:
amostra = X_teste[0]
classe_prevista = prever(modelo, amostra)[0]

print("amostra:", amostra.astype(int))
print("classe prevista:", ORDEM_CLASSES[classe_prevista], "| classe real:", y_teste_letras[0])

amostra: [1 0 0 0 0 1 0 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 0 1 0 0]
classe prevista: G | classe real: G
